In [1]:
import json
import pickle
from pathlib import Path

import numpy as np

import sys
sys.path.append("../workflow/scripts")
from helpers import fit_clorinn, ensure_parent, clorinn_to_dict
from fit_cv_model import load_cv_data, heldout_metrics

In [2]:
cv_input = f"/gpfs/commons/groups/knowles_lab/data/PsychGen/analysis/clorinn/cv_input/zscore_cv.npz"
method = "nnm"
nucnorm = float("8192")
max_iter = 1000
svd_max_iter = 100
model_out = f"/gpfs/commons/groups/knowles_lab/data/PsychGen/analysis/clorinn/cross_validation/models/nnm_cv_model_r128.pkl"
metrics_out = f"/gpfs/commons/groups/knowles_lab/data/PsychGen/analysis/clorinn/cross_validation/metrics/nnm_cv_metrics_r128.json"

In [3]:
ztrue, ztrain, zmask = load_cv_data(cv_input)
ztrain_filled = np.nan_to_num(ztrue, 0)
np.linalg.norm(ztrain_filled, ord = 'nuc')

29849.55340914954

In [4]:
from clorinn.optimize import ProjectedGradientDescent as PGD

# nan_mask = np.isnan(ztrain)
# Y = np.nan_to_num(ztrain, nan=0.0)

# Phase 1: PGD
pgd = PGD(max_iter=50, verbose=2)
pgd.fit(ztrain, nucnorm)
print(f"PGD finished in {pgd.result.n_iter} iterations")

2026-04-27 15:42:36,528 | pgd                      | DEBUG   | Created <Logger clorinn.optimize.pgd (DEBUG)>, Parent: <Logger clorinn (WARNING)>
2026-04-27 15:42:36,529 | pgd                      | DEBUG   | Model = nnm
2026-04-27 15:42:36,529 | pgd                      | DEBUG   | Shape of input data = (107, 46504)
2026-04-27 15:42:36,561 | objectives               | DEBUG   | NNMObjective: shape=(107, 46504), radius=8192.0, masked_entries=721617, weighted=no
2026-04-27 15:42:36,562 | pgd                      | DEBUG   | Masked entries = 721617
2026-04-27 15:42:37,049 | pgd                      | INFO    | PGD iter    1  f = 2227191.2159  ||X||_* = 8192.0  (clipped)
2026-04-27 15:42:37,525 | pgd                      | INFO    | PGD iter    2  f = 2200454.2497  ||X||_* = 8192.0  (clipped)
2026-04-27 15:42:37,989 | pgd                      | INFO    | PGD iter    3  f = 2196734.3490  ||X||_* = 8192.0  (clipped)
2026-04-27 15:42:38,545 | pgd                      | INFO    | PGD iter    4

In [5]:
np.linalg.norm(pgd.result.X, ord='nuc')

8192.000000000038

In [6]:
from clorinn import FrankWolfe, MatrixFactorization
mf = MatrixFactorization(k = 20)
mf.fit(pgd.result.X)
s2 = mf.s**2
np.cumsum(s2)/np.sum(s2)

array([0.22023086, 0.42794757, 0.56123994, 0.63701227, 0.7041738 ,
       0.76152097, 0.81022333, 0.85230446, 0.88382079, 0.90649784,
       0.92599778, 0.94013692, 0.95178969, 0.96299306, 0.97043273,
       0.97690343, 0.98187319, 0.98618147, 0.98946784, 0.99221243,
       0.9947922 , 0.9965341 , 0.99769811, 0.99861584, 0.9989864 ,
       0.99930556, 0.99951494, 0.99968973, 0.9997914 , 0.99987594,
       0.99995794, 0.99998623, 0.99999808, 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.     

In [7]:
from clorinn import FrankWolfe
nnm = FrankWolfe(
    model='nnm', max_iter=200, svd_method='left-gram',
    stop_criteria=['duality_gap'], tol=1e-2,
    print_skip = 100
)
nnm.fit(ztrain, 8192, X0 = pgd.result.X)

2026-04-27 15:42:48,330 | frankwolfe               | INFO    | Iteration 1. Step size 0.000. Duality Gap 1130.8
2026-04-27 15:42:55,694 | frankwolfe               | INFO    | Iteration 100. Step size 0.000. Duality Gap 51.8979
2026-04-27 15:43:03,154 | frankwolfe               | INFO    | Iteration 200. Step size 0.000. Duality Gap 34.8923
2026-04-27 15:43:03,155 | frankwolfe               | INFO    | Stopping at iteration 200. Step size 0.000. Duality Gap 34.8923
2026-04-27 15:43:03,156 | frankwolfe               | INFO    | Maximum number of iterations reached.


In [8]:
np.sum(np.linalg.svd(nnm.result.X, compute_uv=False))

8191.999999333758

In [9]:
from clorinn import FrankWolfe, MatrixFactorization
mf = MatrixFactorization(k = 20)
mf.fit(pgd.result.X)
s2 = mf.s**2
energy = np.cumsum(s2) / np.sum(s2)
print ("X0 after PGD")
print (energy[:20])

mf = MatrixFactorization(k = 20)
mf.fit(nnm.result.X)
s2 = mf.s**2
energy = np.cumsum(s2) / np.sum(s2)
print ("FW after PGD")
print (energy[:20])

X0 after PGD
[0.22023086 0.42794757 0.56123994 0.63701227 0.7041738  0.76152097
 0.81022333 0.85230446 0.88382079 0.90649784 0.92599778 0.94013692
 0.95178969 0.96299306 0.97043273 0.97690343 0.98187319 0.98618147
 0.98946784 0.99221243]
FW after PGD
[0.2202654  0.42797568 0.56126874 0.63704652 0.70420852 0.76155294
 0.81025935 0.85234177 0.88385486 0.90652975 0.92602469 0.94015967
 0.95180822 0.96300848 0.97044573 0.97691322 0.98188152 0.98618771
 0.98947275 0.99221621]


In [10]:
print(f"PGD objective:         {pgd.result.loss:.2f}")
print(f"FW after 1000 steps:   {nnm.result.loss:.2f}")
print(f"Relative improvement:  {(pgd.result.loss - nnm.result.loss) / pgd.result.loss:g}")

PGD objective:         2195713.31
FW after 1000 steps:   2195713.28
Relative improvement:  1.4628e-08


In [11]:
model = fit_clorinn(ztrain, nucnorm, model=method, solver = "pgd", pgd_max_iter = 5)

2026-04-27 15:43:13,669 | pgd                      | INFO    | PGD iter    1  f = 2227191.2159  ||X||_* = 8192.0  (clipped)
2026-04-27 15:43:14,233 | pgd                      | INFO    | PGD iter    2  f = 2200454.2497  ||X||_* = 8192.0  (clipped)
2026-04-27 15:43:14,768 | pgd                      | INFO    | PGD iter    3  f = 2196734.3490  ||X||_* = 8192.0  (clipped)
2026-04-27 15:43:15,286 | pgd                      | INFO    | PGD iter    4  f = 2195968.9558  ||X||_* = 8192.0  (clipped)
2026-04-27 15:43:15,824 | pgd                      | INFO    | PGD iter    5  f = 2195783.2574  ||X||_* = 8192.0  (clipped)


In [12]:
model.history.loss

[6323690.45583295,
 2227191.2158978637,
 2200454.249727821,
 2196734.3489680705,
 2195968.9557587258,
 2195783.2574419263]

In [16]:
model = fit_clorinn(ztrain, nucnorm, model=method, solver = "pgd-fw", max_iter = 200, pgd_max_iter = 5)

2026-04-27 15:44:17,740 | pgd                      | INFO    | PGD iter    1  f = 2227191.2159  ||X||_* = 8192.0  (clipped)
2026-04-27 15:44:18,229 | pgd                      | INFO    | PGD iter    2  f = 2200454.2497  ||X||_* = 8192.0  (clipped)
2026-04-27 15:44:18,746 | pgd                      | INFO    | PGD iter    3  f = 2196734.3490  ||X||_* = 8192.0  (clipped)
2026-04-27 15:44:19,224 | pgd                      | INFO    | PGD iter    4  f = 2195968.9558  ||X||_* = 8192.0  (clipped)
2026-04-27 15:44:19,703 | pgd                      | INFO    | PGD iter    5  f = 2195783.2574  ||X||_* = 8192.0  (clipped)
2026-04-27 15:44:19,896 | frankwolfe               | INFO    | Iteration 1. Step size 0.000. Duality Gap 13824.4
2026-04-27 15:44:21,347 | frankwolfe               | INFO    | Iteration 20. Step size 0.000. Duality Gap 2199.72
2026-04-27 15:44:22,862 | frankwolfe               | INFO    | Iteration 40. Step size 0.000. Duality Gap 1467.12
2026-04-27 15:44:24,375 | frankwolfe   

In [17]:
model.loss

2195777.422767829

In [14]:
nnm.result.history.duality_gap

[inf,
 1130.7991352680892,
 617.3830023439324,
 461.1189593632828,
 428.55523376992255,
 247.17185619237773,
 260.753522002566,
 248.52285729186553,
 281.6922311882739,
 228.503464625059,
 192.23939904116634,
 182.69874179807766,
 177.84187111627944,
 169.52747748484262,
 170.72619479232674,
 178.84292544751202,
 174.9852946931673,
 167.97571421151497,
 147.32596777428955,
 151.59052504413683,
 138.36948185856977,
 139.086899759493,
 128.28701612472722,
 132.6241469754665,
 123.48458894726264,
 116.97960258754938,
 115.91609918215312,
 116.42801453355924,
 113.51687628630498,
 122.39828362061405,
 113.3675088381666,
 113.60392307290283,
 117.1762302079153,
 102.38066635176938,
 98.70206146575168,
 92.48443058404884,
 97.8208628982265,
 98.69241078250298,
 99.40363491004473,
 98.66032147227179,
 100.96309459575104,
 94.05717069424873,
 97.26829322151684,
 87.55680788441418,
 92.15747835931525,
 89.09694469870934,
 87.02139284354269,
 81.48814041514822,
 89.80433791643532,
 96.7173602150

In [16]:
ztrain_filled = np.nan_to_num(ztrain, 0)
np.linalg.norm(nnm.result.X, ord = 'fro') / np.linalg.norm(ztrain, ord = 'fro')

0.5812705480679592

In [9]:
# -- Save results ------
metrics = heldout_metrics(ztrue, model.X, zmask)
metrics.update(
    {
        "method": method,
        "nucnorm": nucnorm,
        "max_iter": max_iter,
        "n_iter" : len(model.steps),
    }
)
model_dict = clorinn_to_dict(model)

ensure_parent(model_out)
ensure_parent(metrics_out)

with open(model_out, "wb") as handle:
    pickle.dump(model_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)

with open(metrics_out, "w") as handle:
    json.dump(metrics, handle, indent=2, sort_keys=True)

print(f"Finished model fit at nucnorm={nucnorm:g}")
print(f"Held-out MSE: {metrics['heldout_mse']:.10f}")

Finished model fit at nucnorm=256
Held-out MSE: 2.8656791800


In [ ]:
from sklearn.utils.extmath import randomized_svd
from clorinn.optimize import TopCompSVD

dg = (X0 - z0_pgd) * ~nan_mask

print (f"Any NaNs: {np.any(np.isnan(dg))}")
print (f"Any Infs: {np.any(np.isinf(dg))}")

u1 = {}
v1_t = {}
for method in ('power', 'direct'):
    svd = TopCompSVD(method = method, max_iter = 2000)
    svd.fit(dg)
    u1[method] = svd.u1
    v1_t[method] = svd.v1_t
    if method == 'power':
        print (f"Number of power iterations: {svd.n_iter}")
        
print (f"u1 norm diff: {np.linalg.norm(u1['power'] - u1['direct'], ord='fro')}")
print (f"v1 norm diff: {np.linalg.norm(v1_t['power'] - v1_t['direct'], ord='fro')}")